In [1]:
import json
import floorplan_backend
import time

In [6]:
with open('./samples/topology.json') as topology_json:
    with open('./samples/construction.json') as construction_json:
        topology = json.load(topology_json)
        construction = json.load(construction_json)
        start = time.time()
        building = floorplan_backend.Building.from_data(topology, construction)
        print(time.time()-start)
        building.solve_joints()
        geoms = {
            uuid: result
            for face_id in building.face_ids()
            for uuid, result in building.layer_geometry(face_id).items()
        }

0.004899024963378906


In [9]:
construction

{'materials': {'0cbab760-7b22-4202-9e56-4d096260f8ad': {'name': 'OSB',
   'density': 600,
   'heat_transfer_coefficient': 0.12,
   'specific_heat_capacity': 1600,
   'dictionary': {}},
  '4ca55fd2-ce4b-4c8b-b169-8c16320cebf4': {'name': 'Fir',
   'density': 400,
   'heat_transfer_coefficient': 0.11,
   'specific_heat_capacity': 1500,
   'dictionary': {}},
  '2cb9546a-e67c-4abc-a232-4a75f5c7d0c0': {'name': 'Mineral wool',
   'density': 50,
   'heat_transfer_coefficient': 0.033,
   'specific_heat_capacity': 1030,
   'dictionary': {}},
  'f1999927-0e5a-4b9e-9f46-b8cae020db32': {'name': 'Ceramic facade tiles',
   'density': 2000,
   'heat_transfer_coefficient': 1,
   'specific_heat_capacity': 1000,
   'dictionary': {}},
  '73dd0c9b-3596-4123-b48e-73cd2a2622fb': {'name': 'Concrete',
   'density': 2200,
   'heat_transfer_coefficient': 1.5,
   'specific_heat_capacity': 840,
   'dictionary': {}}},
 'composites': {'e48e81f5-5daf-4ffd-be13-665bf6da2709': {'name': 'Studs',
   'part1': '4ca55fd2-ce

In [8]:
geoms

{'3790eb76-3f31-4e16-9503-3457ecec08c9': [{'name': 'plate',
   'uuid': '5c7b651d-6557-4643-8043-b240473314c1',
   'area': 5.4561992191558275,
   'joints': [{'face_Id': '850f46b6-ac99-40af-8927-171f6c81a770',
     'layer_idx': 2},
    {'face_Id': '5c13aebb-929e-4fbe-98c4-e404c80d7de8', 'layer_idx': 2},
    {'face_Id': 'e23dc605-4dd1-4ca0-a1f6-76ea43e3bd76', 'layer_idx': 0},
    {'face_Id': 'df9f508e-45a3-4df0-bf37-6d6839a685f0', 'layer_idx': 0}],
   'inner': array([[ 8.00416558,  9.41516003,  6.22      ],
          [ 8.00416558,  9.41516003,  9.        ],
          [ 8.00416558, 11.36075115,  9.        ],
          [ 8.00416558, 11.36075115,  6.22      ]]),
   'outer': array([[ 7.99416558,  9.40516003,  6.21      ],
          [ 7.99416558,  9.40516003,  9.01      ],
          [ 7.99416558, 11.37075115,  9.01      ],
          [ 7.99416558, 11.37075115,  6.21      ]])},
  {'name': 'studs',
   'uuid': '202df3c8-5714-4c40-a83e-181d76104c64',
   'area': 5.783655130335596,
   'joints': [{'fa

In [ ]:
import numpy as np
import plotly.graph_objects as go

display_material = 'studs'

def polygon_mesh(face):
    face = np.asarray(face)
    n = len(face)
    return (face[:, 0],face[:, 1],face[:, 2],np.zeros(n - 2, dtype=int),np.arange(1, n - 1),np.arange(2, n))

fig = go.Figure()

for geom in geoms:
    colour = np.random.randint(0,255,3)
    colour = f"rgb({colour[0]}, {colour[1]}, {colour[2]})"
    
    for layer in geom:
        for face in [layer['inner'],layer['outer']]:
            x, y, z, i, j, k = polygon_mesh(face)
            
            fig.add_trace(go.Mesh3d(x=x,y=y,z=z,i=i,j=j,k=k,opacity=1,flatshading=True,color=colour,name=layer["name"]))

fig.update_layout(scene=dict(aspectmode="data",xaxis_title="X",yaxis_title="Y",zaxis_title="Z",),)
fig.write_html("geometry.html", auto_open=True)
#fig.show()